Note:
- This notebook is part of the Gao et al. (2026, in preparation) analysis workspace:
  Gao et al., Minute-scale coupling of chromatin marks and transcriptional bursts (2026, in preparation)
- For data layout, execution order, and key parameters, see the README.md in this folder:
  - analyses/02_Cross_correlation/README.md
- State labels:
  - Active = ON
  - Inactive = OFF
  Some intermediate tables may use ON/OFF; we treat them as equivalent to Active/Inactive.


# Time-lapse Imaging and Transcription Dynamics Analysis Pipeline
---




# Cell 0
---

import necessary packages (if running in a new environment)

In [ ]:
# Cell 0: Environment Setup & Dependencies

import sys
import os
import json
import math
import shutil
import warnings
import datetime
from pathlib import Path
from dataclasses import dataclass
from typing import Tuple, Optional, List

# Data Science & Math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Image Processing
from tifffile import imread, imwrite
from scipy.ndimage import gaussian_filter, center_of_mass, label
from skimage.measure import label as sk_label, regionprops
from skimage.filters import threshold_otsu
from skimage.morphology import (
    remove_small_objects,
    remove_small_holes,
    binary_opening,
    binary_closing,
    disk,
)
from skimage.measure import regionprops
from skimage.exposure import rescale_intensity

# Specialized Bio-image Analysis
import bigfish.detection as detection
import napari
from magicgui.widgets import PushButton, Container

# Utilities
from tqdm import tqdm

# Configuration
warnings.filterwarnings("ignore")
plt.rcParams['figure.dpi'] = 150
print(f"Environment ready. Python version: {sys.version}")

## Cell 1: Segmentation and Cell Tracking

**Objective**
To segment a single cell (primarily the nuclear region) in each frame and track it over time.

**Choice of segmentation image**  
In this pipeline, segmentation is performed on 2D maximum-intensity projections (MIPs).  
By default, MIPs of the SNAPtag channel are used when SNAPtag expression is relatively uniform across cells—such as in knock-in settings—providing stable and well-defined cell boundaries.
In contrast, when SNAPtag is exogenously expressed and exhibits substantial cell-to-cell variability, or when nuclear boundaries are poorly defined and segmentation becomes unstable, a composite segmentation image is used. This composite image is generated by robustly normalizing the MIPs of the MCP and mTetR channels (1st–99.8th intensity percentiles) and summing the two normalized images to enhance cell-to-background contrast.  
The segmentation source is explicitly specified using the parameter SEGMENTATION_SOURCE (snap, composite, or auto).  
  
**Processing overview**
* Z-max projection (MIP)
* Robust normalization followed by Gaussian smoothing
* Otsu thresholding with morphological opening/closing and removal of small objects
* Centroid-based tracking (with a maximum jump constraint)
  
**Outputs**  
mask.npy and track.csv, with the segmentation source used for each frame recorded.

In [ ]:
# Cell 1: single-cell mask + tracking

# paths 
IN_DIR = Path("../../../data/02_Cross_correlation").expanduser()
OUT_DIR = Path("../outputs/1_analysis/1_masks").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# parameters 
OVERWRITE = False
SAVE_DEBUG = True

# ----------------------------
# Segmentation image source (explicit + reproducible)
# ----------------------------
# "snap"      : use SNAPtag MIP (recommended when SNAPtag provides stable boundaries; e.g., knock-in / low heterogeneity)
# "composite" : use robust-normalized MCP + mTetR MIPs (recommended when SNAPtag is heterogeneous or segmentation is unstable)
# "auto"      : try "snap" first; if no valid region is found, fall back to "composite"
SEGMENTATION_SOURCE = "snap"   # <- change here: "snap" / "composite" / "auto"

def build_segmentation_image(vol_tzc, params: "SegParams", source: str):
    """
    vol_tzc: 4D array with shape (Z, C, Y, X) for a single time point.
    returns: 2D segmentation image (Y, X) in [0,1] (approximately)
    """
    if source == "snap":
        proj = zmax_projection(vol_tzc[:, params.ch_snap])
        return robust_normalize(proj)
    elif source == "composite":
        proj_tetr = zmax_projection(vol_tzc[:, params.ch_tetr])
        proj_mcp  = zmax_projection(vol_tzc[:, params.ch_mcp])
        seg = robust_normalize(proj_tetr) + robust_normalize(proj_mcp)
        # optional: rescale to [0,1] for more stable Otsu behavior
        return np.clip(seg / 2.0, 0, 1)
    else:
        raise ValueError(f"Unknown SEGMENTATION_SOURCE: {source}")


# helper functions 
def robust_normalize(img, p_low=1.0, p_high=99.8):
    img = img.astype(np.float32, copy=False)
    lo = np.percentile(img, p_low)
    hi = np.percentile(img, p_high)
    if hi <= lo + 1e-6:
        return np.zeros_like(img)
    return np.clip((img - lo) / (hi - lo), 0, 1)
# z-projection
def zmax_projection(vol):
    return np.max(vol, axis=0)
# overlay
def make_overlay(rgb, mask, alpha=0.35):
    out = rgb.copy()
    red = np.zeros_like(out)
    red[..., 0] = 1.0
    out[mask] = (1 - alpha) * out[mask] + alpha * red[mask]
    return out

# segmentation 
@dataclass
class SegParams:
    gaussian_sigma: float = 1.0
    min_area: int = 2000
    max_area: int = 60000
    open_radius: int = 2
    close_radius: int = 3
    hole_area: int = 1500
    small_object_area: int = 1500
    ch_snap: int = 0
    ch_mcp: int = 1
    ch_tetr: int = 2

def segment_one_frame(seg_img, params):
    sm = gaussian_filter(seg_img, sigma=params.gaussian_sigma)
    try:
        thr = threshold_otsu(sm)
    except Exception:
        thr = sm.mean() + 2 * sm.std()
    bw = sm > thr
    bw = binary_opening(bw, disk(params.open_radius))
    bw = binary_closing(bw, disk(params.close_radius))
    bw = remove_small_holes(bw, params.hole_area)
    bw = remove_small_objects(bw, params.small_object_area)
    return bw

# tracking (with gating) 
def pick_target_region(
    props, img_shape, prev_centroid=None,
    min_area=2000, max_area=60000,
    max_jump=60.0,
    ref_area=None, area_lo=0.5, area_hi=2.0
):
    if not props:
        return None

    H, W = img_shape
    center = np.array([H / 2, W / 2])

    candidates = []
    for p in props:
        if p.area < min_area or p.area > max_area:
            continue
        if ref_area is not None:
            if not (area_lo * ref_area <= p.area <= area_hi * ref_area):
                continue
        candidates.append((p, np.array(p.centroid)))

    if not candidates:
        return None

    if prev_centroid is None:
        return min(candidates, key=lambda x: np.linalg.norm(x[1] - center))[0]

    prev = np.array(prev_centroid)
    p, dist = min(
        ((p, np.linalg.norm(c - prev)) for p, c in candidates),
        key=lambda x: x[1],
    )
    return p if dist <= max_jump else None

# main processing 
def process_tiff_notebook(tiff_path, out_root, params, overwrite=False, save_debug=True):
    stem = tiff_path.stem
    out_dir = out_root / stem
    (out_dir / "debug").mkdir(parents=True, exist_ok=True)

    mask_path = out_dir / "mask.npy"
    track_path = out_dir / "track.csv"
    meta_path = out_dir / "meta.json"

    if mask_path.exists() and track_path.exists() and meta_path.exists() and not overwrite:
        return

    arr = imread(tiff_path)
    T, Z, C, Y, X = arr.shape

    meta_path.write_text(json.dumps({
        "shape": arr.shape,
        "params": params.__dict__,
    }, indent=2))

    masks = np.zeros((T, Y, X), bool)
    rows = []
    prev_centroid = None
    ref_area = None

    for t in range(T):
        vol_tzc = arr[t]  # (Z, C, Y, X)

        # 1) build segmentation image
        seg_source_used = SEGMENTATION_SOURCE
        seg_img = build_segmentation_image(vol_tzc, params, "snap" if SEGMENTATION_SOURCE == "auto" else SEGMENTATION_SOURCE)

        # 2) segment + pick region
        bw = segment_one_frame(seg_img, params)
        lab = sk_label(bw)
        props = regionprops(lab)
        target = pick_target_region(
            props, (Y, X), prev_centroid,
            params.min_area, params.max_area,
            max_jump=60, ref_area=ref_area
        )

        # 3) auto-fallback: snap -> composite when segmentation fails
        if target is None and SEGMENTATION_SOURCE == "auto":
            seg_img2 = build_segmentation_image(vol_tzc, params, "composite")
            bw2 = segment_one_frame(seg_img2, params)
            lab2 = sk_label(bw2)
            props2 = regionprops(lab2)
            target2 = pick_target_region(
                props2, (Y, X), prev_centroid,
                params.min_area, params.max_area,
                max_jump=60, ref_area=ref_area
            )
            if target2 is not None:
                seg_img = seg_img2
                lab = lab2
                target = target2
                seg_source_used = "composite"
            else:
                seg_source_used = "snap->composite_failed"

        # 4) fallback behavior (unchanged) + record seg_source_used
        if target is None:
            if t > 0 and prev_centroid is not None:
                masks[t] = masks[t-1]
                cy, cx = prev_centroid
                rows.append({
                    "t": t, "found": 0, "fallback": 1,
                    "centroid_y": cy, "centroid_x": cx,
                    "area": int(masks[t].sum()),
                    "seg_source": seg_source_used
                })
            else:
                rows.append({
                    "t": t, "found": 0, "fallback": 0,
                    "centroid_y": np.nan, "centroid_x": np.nan,
                    "area": 0,
                    "seg_source": seg_source_used
                })
        else:
            if ref_area is None:
                ref_area = target.area
            m = (lab == target.label)
            masks[t] = m
            prev_centroid = target.centroid
            rows.append({
                "t": t, "found": 1, "fallback": 0,
                "centroid_y": float(target.centroid[0]),
                "centroid_x": float(target.centroid[1]),
                "area": int(target.area),
                "seg_source": seg_source_used
            })

        if save_debug:
            disp = rescale_intensity(seg_img, out_range=(0, 1))
            rgb = np.stack([disp]*3, -1)
            overlay = make_overlay(rgb, masks[t])
            fig, ax = plt.subplots(1, 3, figsize=(12, 4))
            ax[0].imshow(disp, cmap="gray"); ax[0].set_title(f"t={t} | {seg_source_used}")
            ax[1].imshow(masks[t], cmap="gray")
            ax[2].imshow(overlay)
            for a in ax:
                a.axis("off")
            fig.tight_layout()
            fig.savefig(out_dir / "debug" / f"t{t:02d}.png")
            plt.close(fig)

    np.save(mask_path, masks)
    pd.DataFrame(rows).to_csv(track_path, index=False)

# run 
params = SegParams()
tiffs = sorted([p for p in IN_DIR.iterdir() if p.suffix.lower() in [".tif", ".tiff"]])

print("Found TIFFs:", len(tiffs))
for p in tqdm(tiffs, desc="Processing TIFFs"):
    process_tiff_notebook(p, OUT_DIR, params, overwrite=OVERWRITE, save_debug=SAVE_DEBUG)

print("Done. Outputs at:", OUT_DIR)


## Cell 2: 3D Spot Detection (TetR & MCP)

**Objective:**
To detect diffraction-limited spots for TetR (genomic anchor) and MCP (nascent RNA) in 3D space.

**Methodology:**
- **Input:** Raw TIFFs and masks generated in Part 1.
- **Library:** `Big-FISH` detection algorithm.
- **Parameters:**
    - Voxel size: 300nm (Z) x 130nm (XY).
    - Spot radius: 400nm (Z) x 180nm (XY).
- **Output:** Coordinates and intensities stored in `spots_tetr_mcp.csv`, plus a Fiji-compatible debug hyperstack.

In [ ]:
# Cell 2: 3D Spot Detection (TetR + MCP) using Big-FISH
# 3-channel unchanged; output Fiji-openable 2D hyperstack tif (MIP + overlays in ch3)

# paths 
RAW_DIR = IN_DIR
MASK_DIR = Path("outputs/1_analysis/1_masks").expanduser()
OUT_DIR  = Path("outputs/1_analysis/2_spots").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# parameters 
OVERWRITE = False
VOXEL_SIZE_Z  = 300   # nm
VOXEL_SIZE_YX = 130   # nm
SPOT_RADIUS_Z  = 400  # nm
SPOT_RADIUS_YX = 180  # nm

# channels (0-based)
CH_MCP  = 1  # channel2
CH_TETR = 2  # channel3

# helper functions 
def get_best_spot(spots, scores, prev_coords=None, max_jump_pixels=30):
    if spots is None or len(spots) == 0:
        return None, None

    sorted_indices = np.argsort(scores)[::-1]

    if prev_coords is None:
        best_idx = sorted_indices[0]
        return spots[best_idx], scores[best_idx]

    prev_z, prev_y, prev_x = prev_coords
    for idx in sorted_indices:
        curr = spots[idx]
        dist = np.linalg.norm(curr[1:] - np.array([prev_y, prev_x]))
        if dist <= max_jump_pixels:
            return curr, scores[idx]

    best_idx = sorted_indices[0]
    return spots[best_idx], scores[best_idx]


def draw_x(img2d, y, x, value, size=4):
    """Draw an 'X' marker on a 2D image at (y,x)."""
    if y is None or x is None:
        return
    if np.isnan(y) or np.isnan(x):
        return
    y = int(y); x = int(x)
    H, W = img2d.shape
    for d in range(-size, size + 1):
        y1, x1 = y + d, x + d
        y2, x2 = y + d, x - d
        if 0 <= y1 < H and 0 <= x1 < W:
            img2d[y1, x1] = value
        if 0 <= y2 < H and 0 <= x2 < W:
            img2d[y2, x2] = value


def detect_one_channel(vol_3d, prev_coords):
    """Run bigfish detection on (Z,Y,X) volume and select best spot."""
    try:
        spots, thr = detection.detect_spots(
            images=vol_3d,
            return_threshold=True,
            voxel_size=(VOXEL_SIZE_Z, VOXEL_SIZE_YX, VOXEL_SIZE_YX),
            spot_radius=(SPOT_RADIUS_Z, SPOT_RADIUS_YX, SPOT_RADIUS_YX)
        )
        if len(spots) > 0:
            intens = vol_3d[spots[:, 0], spots[:, 1], spots[:, 2]]
        else:
            intens = []
    except Exception as e:
        return None, None, str(e)

    best, score = get_best_spot(spots, intens, prev_coords)
    return best, score, None

# main processing per file
def process_single_file(tiff_path, mask_path, out_root, overwrite=False):
    stem = tiff_path.stem
    file_out_dir = out_root / stem
    file_out_dir.mkdir(parents=True, exist_ok=True)

    csv_path = file_out_dir / "spots_tetr_mcp.csv"
    debug_tif_path = file_out_dir / "debug_mip_overlay.tif"

    if csv_path.exists() and (not overwrite):
        return

    if not mask_path.exists():
        print(f"Skipping {stem}, no mask found.")
        return

    img_5d = imread(tiff_path)      # expected: (T, Z, C, Y, X)
    masks_3d = np.load(mask_path)   # expected: (T, Y, X)

    if img_5d.ndim != 5:
        print(f"Skipping {stem}, unexpected tiff ndim={img_5d.ndim}, shape={img_5d.shape}")
        return

    T, Z_dim, C, Y, X = img_5d.shape
    if masks_3d.shape[0] != T:
        print(f"Skipping {stem}, mask T={masks_3d.shape[0]} != img T={T}")
        return

    # marker intensities (3 channels unchanged; marks are written into ch3 only)
    if np.issubdtype(img_5d.dtype, np.integer):
        maxv = np.iinfo(img_5d.dtype).max
        tetr_mark = maxv
        mcp_mark = int(maxv * 0.6)
    else:
        tetr_mark = 1.0
        mcp_mark = 0.6

    spot_data = []
    prev_tetr = None
    prev_mcp = None

    # debug hyperstack: (T, C, Y, X) = 2D MIP per channel per frame
    debug_stack = np.zeros((T, C, Y, X), dtype=img_5d.dtype)

    for t in range(T):
        mask_t = masks_3d[t]

        # --- build 2D MIPs for all channels (debug base) ---
        for c in range(C):
            debug_stack[t, c] = img_5d[t, :, c].max(axis=0)

        # --- TetR detection on 3D ---
        vol_tetr = img_5d[t, :, CH_TETR].copy()  # (Z,Y,X)
        vol_tetr[:, ~mask_t] = 0
        best_tetr, score_tetr, err = detect_one_channel(vol_tetr, prev_tetr)
        if err is not None:
            print(f"[{stem}] TetR detection error t={t}: {err}")
            best_tetr, score_tetr = None, None

        # --- MCP detection on 3D ---
        vol_mcp = img_5d[t, :, CH_MCP].copy()
        vol_mcp[:, ~mask_t] = 0
        best_mcp, score_mcp, err = detect_one_channel(vol_mcp, prev_mcp)
        if err is not None:
            print(f"[{stem}] MCP detection error t={t}: {err}")
            best_mcp, score_mcp = None, None

        row = {"t": t}

        if best_tetr is not None:
            row.update({
                "tetr_z": int(best_tetr[0]), "tetr_y": int(best_tetr[1]), "tetr_x": int(best_tetr[2]),
                "tetr_intensity": float(score_tetr), "tetr_detected": True
            })
            prev_tetr = tuple(best_tetr)
        else:
            row.update({
                "tetr_z": np.nan, "tetr_y": np.nan, "tetr_x": np.nan,
                "tetr_intensity": np.nan, "tetr_detected": False
            })
            prev_tetr = None

        if best_mcp is not None:
            row.update({
                "mcp_z": int(best_mcp[0]), "mcp_y": int(best_mcp[1]), "mcp_x": int(best_mcp[2]),
                "mcp_intensity": float(score_mcp), "mcp_detected": True
            })
            prev_mcp = tuple(best_mcp)
        else:
            row.update({
                "mcp_z": np.nan, "mcp_y": np.nan, "mcp_x": np.nan,
                "mcp_intensity": np.nan, "mcp_detected": False
            })
            prev_mcp = None

        spot_data.append(row)

        # --- draw markers ONLY into channel3 (CH_TETR) MIP ---
        ch3_mip = debug_stack[t, CH_TETR]
        if best_tetr is not None:
            draw_x(ch3_mip, best_tetr[1], best_tetr[2], tetr_mark, size=4)

        #MCP draw in ch2
        ch2_mip = debug_stack[t, CH_MCP]
        if best_mcp is not None:
            draw_x(ch2_mip, best_mcp[1], best_mcp[2], mcp_mark, size=4)

    # save CSV
    pd.DataFrame(spot_data).to_csv(csv_path, index=False)

    # save Fiji-openable 2D hyperstack tif (T,C,Y,X)
    imwrite(
        debug_tif_path,
        debug_stack,
        imagej=True,
        metadata={"axes": "TCYX"}
    )


# Main Loop 
if __name__ == "__main__":
    tiffs = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in [".tif", ".tiff"]])
    print(f"Processing {len(tiffs)} files for TetR+MCP spot detection...")

    for tiff_file in tqdm(tiffs):
        mask_file = MASK_DIR / tiff_file.stem / "mask.npy"
        process_single_file(tiff_file, mask_file, OUT_DIR, overwrite=OVERWRITE)

    print(f"\nDone! Results saved to {OUT_DIR}")

## Cell 3: Interactive Manual Verification

**Objective:**
To manually verify and correct false positives/negatives in spot detection using a GUI.

**Workflow:**
- **Tool:** Napari Viewer.
- **Interaction:**
    - Load 5D images and detected points.
    - Users can add/delete points for TetR (Channel 3) and MCP (Channel 2).
    - **"Save & Next"** button computes the optimal Z-plane for manually added points and saves the corrected dataset.
- **Output:** `corrected_spots.csv`.

In [ ]:
# Cell 3: Manual Check (2D Logic) - TetR + MCP

# path 

IN_DIR  = Path("outputs/1_analysis/2_spots").expanduser()
OUT_DIR = Path("outputs/1_analysis/3_corrected_spots").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# channls
CH_MCP  = 1   # Channel 2 (Red)
CH_TETR = 2  # Channel 3 (Green)

LAYER_TETR = "TetR_Spots"
LAYER_MCP  = "MCP_Spots"
# Napari UI + Logic
class SpotChecker:
    def __init__(self, viewer):
        self.viewer = viewer
        self.files = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in [".tif", ".tiff"]])
        self.current_idx = 0
        self.total_files = len(self.files)

        # UI buttons
        self.btn_clear_tetr = PushButton(text="Clear all TetR spots")
        self.btn_clear_tetr.clicked.connect(self.clear_tetr)

        self.btn_clear_mcp = PushButton(text="Clear all MCP spots")
        self.btn_clear_mcp.clicked.connect(self.clear_mcp)

        self.btn_next = PushButton(text="Save & Next Cell")
        self.btn_next.clicked.connect(self.save_and_next)

        # Add buttons to napari dock
        self.viewer.window.add_dock_widget(
            Container(widgets=[self.btn_clear_tetr, self.btn_clear_mcp, self.btn_next]),
            area="right"
        )

        self.load_sample(0)
    # load sample
    def load_sample(self, idx):
        if idx >= self.total_files:
            print("All files processed!")
            self.viewer.layers.clear()
            self.btn_next.enabled = False
            self.btn_clear_tetr.enabled = False
            self.btn_clear_mcp.enabled = False
            return

        self.current_file = self.files[idx]
        stem = self.current_file.stem
        print(f"[{idx+1}/{self.total_files}] Opening {stem}...")
        
        self.viewer.layers.clear()

        # load 5D image
        self.img_5d = imread(self.current_file)

        # 2D MIP projections
        mcp_proj  = np.max(self.img_5d[:, :, CH_MCP,  :, :], axis=1)
        tetr_proj = np.max(self.img_5d[:, :, CH_TETR, :, :], axis=1)

        # add to viewer
        self.viewer.add_image(mcp_proj,  name='Ch2_MCP',  colormap='red',   blending='additive')
        self.viewer.add_image(tetr_proj, name='Ch3_TetR', colormap='green', blending='additive')
        # load spots
        csv_path_new = IN_DIR / stem / "spots_tetr_mcp.csv"
        csv_path_old = IN_DIR / stem / "spots_tetr.csv"
        # load existing spots if any
        tetr_points = np.empty((0, 3), dtype=float)  # [t,y,x]
        mcp_points  = np.empty((0, 3), dtype=float)
        # load from new format first
        if csv_path_new.exists():
            df = pd.read_csv(csv_path_new)

            # TetR
            if "tetr_detected" in df.columns:
                df_t = df[df["tetr_detected"] == True]
                if not df_t.empty:
                    tetr_points = df_t[["t", "tetr_y", "tetr_x"]].rename(
                        columns={"tetr_y": "y", "tetr_x": "x"}
                    )[["t", "y", "x"]].to_numpy()

            # MCP
            if "mcp_detected" in df.columns:
                df_m = df[df["mcp_detected"] == True]
                if not df_m.empty:
                    mcp_points = df_m[["t", "mcp_y", "mcp_x"]].rename(
                        columns={"mcp_y": "y", "mcp_x": "x"}
                    )[["t", "y", "x"]].to_numpy()

        elif csv_path_old.exists():
            df = pd.read_csv(csv_path_old)
            df = df[df["detected"] == True]
            if not df.empty:
                tetr_points = df[["t", "y", "x"]].to_numpy()
        #add points layers
        self.viewer.add_points(
            tetr_points,
            name=LAYER_TETR,
            size=10,
            symbol='cross',
            face_color='blue',     # blue for TetR
            n_dimensional=False
        )

        self.viewer.add_points(
            mcp_points,
            name=LAYER_MCP,
            size=10,
            symbol='cross',
            face_color='red',
            n_dimensional=False
        )

        self.viewer.title = f"Checking: {stem}"
        self.viewer.dims.set_point(0, 0)

    # clear functions
    def clear_tetr(self):
        if LAYER_TETR in self.viewer.layers:
            self.viewer.layers[LAYER_TETR].data = np.empty((0, 3), dtype=float)
    # clear MCP
    def clear_mcp(self):
        if LAYER_MCP in self.viewer.layers:
            self.viewer.layers[LAYER_MCP].data = np.empty((0, 3), dtype=float)

    # compute z from points layer
    def _compute_z_for_layer(self, layer_name, channel_idx, prefix):
        """从 points layer (t,y,x) 回溯 5D 图像求 z, 并返回 list[dict]."""
        if layer_name not in self.viewer.layers:
            return []

        layer_data = self.viewer.layers[layer_name].data
        final_spots = []

        T_limit, Z_limit, _, Y_limit, X_limit = self.img_5d.shape

        for p in layer_data:
            t, y, x = p
            t_idx = int(round(t))
            y_idx = int(round(y))
            x_idx = int(round(x))

            t_idx = np.clip(t_idx, 0, T_limit - 1)
            y_idx = np.clip(y_idx, 0, Y_limit - 1)
            x_idx = np.clip(x_idx, 0, X_limit - 1)

            z_profile = self.img_5d[t_idx, :, channel_idx, y_idx, x_idx]
            best_z = int(np.argmax(z_profile))
            intensity = float(z_profile[best_z])

            final_spots.append({
                "t": t_idx,
                f"{prefix}_z": best_z,
                f"{prefix}_y": y_idx,
                f"{prefix}_x": x_idx,
                f"{prefix}_intensity": intensity,
                f"{prefix}_detected": True
            })

        return final_spots
    # save and next
    def save_and_next(self):
        stem = self.current_file.stem


        tetr_spots = self._compute_z_for_layer(LAYER_TETR, CH_TETR, "tetr")
        mcp_spots  = self._compute_z_for_layer(LAYER_MCP,  CH_MCP,  "mcp")


        df_t = pd.DataFrame(tetr_spots)
        df_m = pd.DataFrame(mcp_spots)

        if df_t.empty and df_m.empty:
            out_df = pd.DataFrame(columns=[
                "t",
                "tetr_z","tetr_y","tetr_x","tetr_intensity","tetr_detected",
                "mcp_z","mcp_y","mcp_x","mcp_intensity","mcp_detected"
            ])
        elif df_t.empty:
            out_df = df_m.sort_values("t")
        elif df_m.empty:
            out_df = df_t.sort_values("t")
        else:
            out_df = pd.merge(df_t, df_m, on="t", how="outer").sort_values("t")

        # save
        save_dir = OUT_DIR / stem
        save_dir.mkdir(parents=True, exist_ok=True)
        out_df.to_csv(save_dir / "corrected_spots.csv", index=False)

        print(f"Saved {stem}: TetR={len(df_t)} points, MCP={len(df_m)} points.")

        # next
        self.current_idx += 1
        self.load_sample(self.current_idx)

# Run Napari Viewer
if __name__ == "__main__":
    viewer = napari.Viewer()
    checker = SpotChecker(viewer)
    napari.run()

## Cell 4: Spot Refinement and Mask Optimization

**Objective:**
To refine the spatial precision of manually corrected spots and regenerate cell masks anchored to the TetR signal.

**Methodology:**
- **Spot Refinement:** Local 3D center-of-mass adjustment around the manual coordinates.
- **Mask Refinement:** - Generates a "Spot-Guided Mask" to ensure the mask always encloses the TetR signal.
    - Uses intensity fallback from the previous frame to prevent segmentation flickering.
- **Output:** Refined masks (`mask.npy`) and high-precision spot coordinates (`refined_tetr_spots.csv`).

In [ ]:
# Cell 4: Refinement (TetR-anchored; regenerate mask + refine TetR spots)

# paths
IN_SPOTS_DIR = Path("outputs/1_analysis/3_corrected_spots").expanduser()
OUT_DIR = Path("outputs/1_analysis/4_refined").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# parameters
CH_TETR = 2   # Green (channel3)

REFINE_RADIUS = 3
BLUR_SIGMA = 2.0
MORPH_RADIUS = 3

# helper functions

def refine_spot_local(img_3d, z, y, x, radius=3):
    z, y, x = int(z), int(y), int(x)
    z_min, z_max = max(0, z - 1), min(img_3d.shape[0], z + 2)
    y_min, y_max = max(0, y - radius), min(img_3d.shape[1], y + radius + 1)
    x_min, x_max = max(0, x - radius), min(img_3d.shape[2], x + radius + 1)

    crop = img_3d[z_min:z_max, y_min:y_max, x_min:x_max]
    if crop.size == 0:
        return float(z), float(y), float(x)

    try:
        bg = crop.min()
        dz, dy, dx = center_of_mass(crop - bg)
        return float(z_min + dz), float(y_min + dy), float(x_min + dx)
    except:
        return float(z), float(y), float(x)

# segmentation function
def get_spot_guided_mask(img_2d, spot_y, spot_x, has_spot):

    img_blur = gaussian_filter(img_2d, sigma=BLUR_SIGMA)

    try:
        thresh = threshold_otsu(img_blur)
    except:
        return np.zeros_like(img_2d, dtype=bool)

    binary = img_blur > thresh
    binary = binary_closing(binary, disk(MORPH_RADIUS))
    binary = remove_small_holes(binary, area_threshold=200)

    labeled, num_features = label(binary)
    if num_features == 0:
        return binary

    target_label = None

    if has_spot and (not np.isnan(spot_y)) and (not np.isnan(spot_x)):
        sy, sx = int(round(spot_y)), int(round(spot_x))
        if 0 <= sy < labeled.shape[0] and 0 <= sx < labeled.shape[1]:
            val = labeled[sy, sx]
            if val > 0:
                target_label = val
            else:
                y_min, y_max = max(0, sy-5), min(labeled.shape[0], sy+6)
                x_min, x_max = max(0, sx-5), min(labeled.shape[1], sx+6)
                local_patch = labeled[y_min:y_max, x_min:x_max]
                counts = np.bincount(local_patch.flatten())
                if len(counts) > 1:
                    target_label = np.argmax(counts[1:]) + 1

    if target_label is None:
        counts = np.bincount(labeled.flatten())
        if len(counts) > 1:
            target_label = np.argmax(counts[1:]) + 1

    if target_label is not None:
        return labeled == target_label
    else:
        return np.zeros_like(img_2d, dtype=bool)
# debug image saving
def save_debug_image(save_path, img_show, mask_bool, spot_y, spot_x, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img_show, cmap='gray')
    if mask_bool is not None and np.any(mask_bool):
        ax.contour(mask_bool, colors='yellow', linewidths=1, alpha=0.7)
    if (spot_y is not None) and (not np.isnan(spot_y)):
        ax.scatter(spot_x, spot_y, c='red', marker='x', s=80, linewidth=2)
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=100)
    plt.close(fig)

# main processing function

def process_refinement():
    spot_folders = sorted([p for p in IN_SPOTS_DIR.iterdir() if p.is_dir()])
    print(f"Found {len(spot_folders)} cells to refine.")

    for spot_folder in tqdm(spot_folders, desc="Processing"):
        stem = spot_folder.name
        raw_path = RAW_DIR / f"{stem}.tif"
        csv_path = spot_folder / "corrected_spots.csv"

        if (not raw_path.exists()) or (not csv_path.exists()):
            continue

        cell_out_dir = OUT_DIR / stem
        debug_dir = cell_out_dir / "debug_refine"
        if debug_dir.exists():
            shutil.rmtree(debug_dir)
        debug_dir.mkdir(parents=True, exist_ok=True)

        img_5d = imread(raw_path)  # (T,Z,C,Y,X)
        spots_df = pd.read_csv(csv_path)
        T = img_5d.shape[0]
        final_masks = np.zeros((T, img_5d.shape[3], img_5d.shape[4]), dtype=bool)
        final_spots = []

        last_mask = None
        # per-frame processing
        for t in range(T):
            # 1) 2D MIP of TetR
            tetr_2d = np.max(img_5d[t, :, CH_TETR], axis=0)
            # 2) normalize
            img_seg = tetr_2d / (tetr_2d.max() + 1e-6)

            # 3) get spot info
            row = spots_df[spots_df["t"] == t]

            refined_z, refined_y, refined_x = np.nan, np.nan, np.nan
            has_spot = False
            intensity_out = np.nan

            # 4) refine spot position
            #  check TetR spot first
            # then fall back to original detected spot
            if (not row.empty):
                if ("tetr_detected" in row.columns) and bool(row.iloc[0].get("tetr_detected", False)):
                    orig_z = row.iloc[0].get("tetr_z", np.nan)
                    orig_y = row.iloc[0].get("tetr_y", np.nan)
                    orig_x = row.iloc[0].get("tetr_x", np.nan)

                    if np.isnan(orig_z) or np.isnan(orig_y) or np.isnan(orig_x):
                        has_spot = False
                        intensity_out = np.nan
                    else:
                        intensity_out = row.iloc[0].get("tetr_intensity", np.nan)
                        vol_tetr = img_5d[t, :, CH_TETR]
                        rz, ry, rx = refine_spot_local(vol_tetr, orig_z, orig_y, orig_x, REFINE_RADIUS)
                        refined_z, refined_y, refined_x = rz, ry, rx
                        has_spot = True

                elif ("detected" in row.columns) and bool(row.iloc[0].get("detected", False)):
                    orig_z = row.iloc[0]["z"]
                    orig_y = row.iloc[0]["y"]
                    orig_x = row.iloc[0]["x"]
                    intensity_out = row.iloc[0].get("intensity", np.nan)

                    vol_tetr = img_5d[t, :, CH_TETR]
                    rz, ry, rx = refine_spot_local(vol_tetr, orig_z, orig_y, orig_x, REFINE_RADIUS)
                    refined_z, refined_y, refined_x = rz, ry, rx
                    has_spot = True

            # save spot info
            if has_spot:
                final_spots.append({
                    "t": t,
                    "z": refined_z,
                    "y": refined_y,
                    "x": refined_x,
                    "detected": True,
                    "intensity": intensity_out
                })
            else:
                final_spots.append({
                    "t": t,
                    "z": np.nan,
                    "y": np.nan,
                    "x": np.nan,
                    "detected": False,
                    "intensity": np.nan
                })

            # 4) generate mask
            current_mask = get_spot_guided_mask(img_seg, refined_y, refined_x, has_spot)

            # fallback to last mask if current is empty
            if current_mask.sum() == 0 and last_mask is not None:
                current_mask = last_mask

            if current_mask.sum() > 0:
                last_mask = current_mask

            final_masks[t] = current_mask

            # 5) save debug image
            save_debug_image(
                debug_dir / f"t{t:02d}.png",
                img_seg,
                current_mask,
                refined_y, refined_x,
                f"{stem} | T={t}"
            )

        # save outputs
        cell_out_dir.mkdir(parents=True, exist_ok=True)
        np.save(cell_out_dir / "mask.npy", final_masks)
        pd.DataFrame(final_spots).to_csv(cell_out_dir / "refined_tetr_spots.csv", index=False)

    print(f"Done. Outputs saved to: {OUT_DIR}")

if __name__ == "__main__":
    process_refinement()

## Cell 5: Transcription Activity Detection (Burst Analysis)

**Objective:**
To determine the transcriptional state (ON/OFF) of the gene locus for each time frame.

**Logic:**
1.  **Anchor:** Uses the refined TetR spot location.
2.  **Colocalization:** Searches for an MCP spot within a defined distance (`DIST_THRESHOLD = 3.0` pixels) from the TetR anchor.
3.  **Intensity Gating:** - Calculates the Signal-to-Noise Ratio (SNR) comparing the MCP spot intensity to the local background.
    - If `Intensity > 1.2 * Background`, the state is classified as **ON**.
- **Output:** `transcription_status.csv` containing binary state flags and intensity metrics.

In [ ]:
# Cell 5: Transcription Activity Detection (MCP Spot) - minimal robust version

# paths

REFINED_DIR = Path("outputs/1_analysis/4_refined").expanduser()
OUT_DIR = Path("outputs/1_analysis/5_transcription").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# parameters
CH_SNAP = 0   
CH_MCP  = 1   
CH_TETR = 2   

DIST_THRESHOLD = 3.0      
ROI_R_YX = 3              
ROI_R_Z  = 2              

MCP_SNR_FACTOR = 1.2 

# Big-FISH parameters (like Cell 2)
VOXEL_SIZE = (300, 130, 130)
SPOT_RADIUS = (400, 180, 180)

# helper functions, In vol(z,y,x), get mean in cropped cube
def crop_mean(vol, z, y, x, rz=1, ryx=3):

    if np.isnan(z) or np.isnan(y) or np.isnan(x):
        return np.nan
    zz, yy, xx = int(round(z)), int(round(y)), int(round(x))
    Z, Y, X = vol.shape
    z0, z1 = max(0, zz - rz), min(Z, zz + rz + 1)
    y0, y1 = max(0, yy - ryx), min(Y, yy + ryx + 1)
    x0, x1 = max(0, xx - ryx), min(X, xx + ryx + 1)
    patch = vol[z0:z1, y0:y1, x0:x1]
    if patch.size == 0:
        return np.nan
    return float(np.mean(patch))

# get local max in 3x3x3 cube
def local_max_intensity(vol, z, y, x):
    if np.isnan(z) or np.isnan(y) or np.isnan(x):
        return np.nan
    zz, yy, xx = map(int, np.round([z, y, x]))
    Z, Y, X = vol.shape
    z0, z1 = max(0, zz-1), min(Z, zz+2)
    y0, y1 = max(0, yy-1), min(Y, yy+2)
    x0, x1 = max(0, xx-1), min(X, xx+2)
    patch = vol[z0:z1, y0:y1, x0:x1]
    if patch.size == 0:
        return np.nan
    return float(np.max(patch))

# main processing function
def process_transcription():
    cell_folders = sorted([p for p in REFINED_DIR.iterdir() if p.is_dir()])

    for cell_folder in tqdm(cell_folders, desc="Analyzing Transcription"):
        stem = cell_folder.name
        raw_path = RAW_DIR / f"{stem}.tif"

        # refined spots path
        spot_path_new = cell_folder / "refined_tetr_spots.csv"
        spot_path_old = cell_folder / "refined_spots.csv"
        spot_path = spot_path_new if spot_path_new.exists() else spot_path_old

        mask_path = cell_folder / "mask.npy"

        if not (raw_path.exists() and spot_path.exists() and mask_path.exists()):
            continue

        img_5d = imread(raw_path)            
        tetr_df = pd.read_csv(spot_path)      
        masks_3d = np.load(mask_path)         

        T = img_5d.shape[0]
        results = []
        has_any_on = False

        # per-frame analysis
        for t in range(T):
            mask_t = masks_3d[t]
            vol_mcp  = img_5d[t, :, CH_MCP,  :, :].copy()
            vol_snap = img_5d[t, :, CH_SNAP, :, :].copy()
            # apply mask
            vol_mcp[:,  ~mask_t] = 0
            vol_snap[:, ~mask_t] = 0
            # get TetR spot info
            row = tetr_df[tetr_df["t"] == t]
            if row.empty:
                results.append({
                    "t": t,
                    "state": "off",
                    "is_on": 0,

                    "tetr_z": np.nan, "tetr_y": np.nan, "tetr_x": np.nan,
                    "tetr_intensity": np.nan,

                    "mcp_z": np.nan, "mcp_y": np.nan, "mcp_x": np.nan,
                    "mcp_intensity": np.nan,

                    "dist": np.nan,
                    "mcp_roi_mean": np.nan,
                    "snap_roi_mean": np.nan
                })
                continue

            t_row = row.iloc[0]
            tetr_detected = bool(t_row.get("detected", False))

            tetr_z = t_row["z"] if tetr_detected else np.nan
            tetr_y = t_row["y"] if tetr_detected else np.nan
            tetr_x = t_row["x"] if tetr_detected else np.nan
            tetr_intensity = t_row.get("intensity", np.nan) if tetr_detected else np.nan

            # TetR surrounding ROI means
            mcp_roi_mean  = crop_mean(vol_mcp,  tetr_z, tetr_y, tetr_x, rz=ROI_R_Z, ryx=ROI_R_YX) if tetr_detected else np.nan
            snap_roi_mean = crop_mean(vol_snap, tetr_z, tetr_y, tetr_x, rz=ROI_R_Z, ryx=ROI_R_YX) if tetr_detected else np.nan

            # default off state
            is_on = 0
            state = "off"
            min_dist = np.nan
            best_mcp = [np.nan, np.nan, np.nan]
            best_mcp_intensity = np.nan

            if tetr_detected:
                # 1) MCP spot detection
                spots_mcp = detection.detect_spots(
                    images=vol_mcp,
                    voxel_size=VOXEL_SIZE,
                    spot_radius=SPOT_RADIUS
                )

                if len(spots_mcp) > 0:
                    tetr_coord = np.array([tetr_z, tetr_y, tetr_x], dtype=float)
                    dists = np.linalg.norm(spots_mcp - tetr_coord, axis=1)
                    min_idx = int(np.argmin(dists))
                    min_dist = float(dists[min_idx])

                    # 2) select best MCP spot within threshold
                    if min_dist <= DIST_THRESHOLD:
                        best_mcp = spots_mcp[min_idx]
                        best_mcp_intensity = local_max_intensity(vol_mcp, best_mcp[0], best_mcp[1], best_mcp[2])

                        # 3) decide on/off
                        if (not np.isnan(best_mcp_intensity)) and (not np.isnan(mcp_roi_mean)) and (mcp_roi_mean > 0):
                            if best_mcp_intensity >= MCP_SNR_FACTOR * mcp_roi_mean:
                                is_on = 1
                                state = "on"
                                has_any_on = True
                        # else remain off

            results.append({
                "t": t,
                "state": state,
                "is_on": int(is_on),

                "tetr_z": tetr_z, "tetr_y": tetr_y, "tetr_x": tetr_x,
                "tetr_intensity": tetr_intensity,

                "mcp_z": best_mcp[0], "mcp_y": best_mcp[1], "mcp_x": best_mcp[2],
                "mcp_intensity": best_mcp_intensity,

                "dist": min_dist,
                "mcp_roi_mean": mcp_roi_mean,
                "snap_roi_mean": snap_roi_mean
            })

        # save only if any "on" detected
        if has_any_on:
            cell_out_dir = OUT_DIR / stem
            cell_out_dir.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(results).to_csv(cell_out_dir / "transcription_status.csv", index=False)

    print(f"Analysis done. Active cells saved to {OUT_DIR}")

if __name__ == "__main__":
    process_transcription()

## Cell 6: Bleaching Correction and Visualization

**Objective:**
Correct photobleaching in intensity traces and generate summary visualizations.

Notes: SNAP (ch1) bleaching correction is always applied using the nuclear mean intensity as a reference. MCP (ch2) bleaching correction is optional (default: auto) and is applied only when bleaching is detectable (recommended for 2-min interval datasets).

**Analysis Steps:**
1.  **Reference extraction:** For each frame, compute a stable reference intensity (SNAP: nuclear mean; MCP: nuclear/cell-mask mean using the same masks).
2.  **Bleaching model fit:** Fit an exponential decay model per channel and compute multiplicative correction factors (with a cap to avoid over-correction).
3.  **Apply correction:** Apply the correction factors to spot-centered ROI mean intensities (snap_roi_mean, mcp_roi_mean) to obtain corrected traces.
4.  **Visualization and aggregation:** Generate per-cell QC plots (raw vs corrected reference) and per-cell corrected traces; aggregate corrected data into global ON/OFF summaries (heatmaps and radial profiles).

- **Output:** all_on.csv, all_off.csv, and merged all_cells_spot_intensity_seg_cell.csv where ch1_mean and ch2_mean are bleach-corrected ROI means.

In [ ]:

# Cell 6: Bleaching Correction and Visualization (SNAP + optional MCP)
# -----------------------------------------------------------------------------
# This script is designed to be a drop-in replacement for the "Cell 6" section
# in Timelapse_original.pdf.
#
# Outputs:
#   1_analysis/6_transcription_summary/
#     ├─ per_cell_plots/*.png
#     ├─ global/*.png
#     ├─ all_on.csv
#     ├─ all_off.csv
#     └─ all_cells_spot_intensity_seg_cell.csv
#
# The merged CSV contains BOTH:
#   - ch1_mean/ch2_mean/ch3_mean (RAW)
#   - ch1_detrended/ch2_detrended (photobleaching-corrected; "detrended" naming kept for compatibility)
#
# NOTE:
#   In downstream analysis (Timelapse_original "A1" onward), the code can pick
#   either *_mean or *_detrended depending on SOURCE_MODE, and will skip the
#   rolling detrend when *_detrended is chosen unless FORCE_DETREND=True.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tifffile import imread
from tqdm import tqdm

from skimage.filters import threshold_otsu
from scipy.ndimage import label as ndi_label


# --------------------------
# Paths (EDIT as needed)
# --------------------------

REFINED_DIR = Path("outputs/1_analysis/4_refined").expanduser()
TRANS_DIR = Path("outputs/1_analysis/5_transcription").expanduser()

OUT_DIR = Path("outputs/1_analysis/6_transcription_summary").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "per_cell_plots").mkdir(exist_ok=True)
(OUT_DIR / "global").mkdir(exist_ok=True)

# --------------------------
# Channels
# --------------------------
CH_SNAP = 0   # ch1
CH_MCP  = 1   # ch2
CH_TETR = 2   # ch3

# --------------------------
# Settings
# --------------------------
T_EXPECT = 60                 # expected frames (0..59)
PATCH_R = 12                  # patch radius in pixels (size = 2*R+1)
EPS = 1e-8

# --------------------------
# Bleaching correction options
# --------------------------
# APPLY_MCP_BLEACH_CORR:
#   - True  : always correct MCP
#   - False : never correct MCP
#   - "auto": correct MCP only when its reference mean shows clear decay
APPLY_MCP_BLEACH_CORR = "auto"
MCP_DECAY_THRESHOLD = 0.10       # "auto": apply if >10% decay (head/tail ratio >= 1.10)
MAX_CORR_FACTOR = 3.0            # cap to avoid over-correction
ENFORCE_DECAY_ONLY = True        # if True, do NOT apply "correction" when fit indicates increasing intensity


# --------------------------
# Helper functions
# --------------------------
def get_nucleus_mask(snap_2d: np.ndarray, cell_mask_2d: np.ndarray) -> np.ndarray:
    """Create a nucleus-like mask from SNAP 2D MIP within the cell mask (bool 2D).
    Uses Otsu thresholding and keeps the largest connected component.
    """
    cell_mask_2d = cell_mask_2d.astype(bool)
    if cell_mask_2d.sum() < 50:
        return np.zeros_like(cell_mask_2d, dtype=bool)

    region = snap_2d[cell_mask_2d]
    try:
        thr = threshold_otsu(region)
    except Exception:
        return np.zeros_like(cell_mask_2d, dtype=bool)

    binary = (snap_2d > thr) & cell_mask_2d
    lab, n = ndi_label(binary)
    if n == 0:
        return np.zeros_like(cell_mask_2d, dtype=bool)

    counts = np.bincount(lab.ravel())
    counts[0] = 0
    k = int(np.argmax(counts))
    return lab == k


def fit_exponential_decay(timepoints: np.ndarray, y: np.ndarray, *, enforce_decay_only: bool = True) -> np.ndarray:
    """Fit y ≈ exp(a + b t) using log-linear regression on finite positive points.
    Returns fitted values at all timepoints.

    If enforce_decay_only=True and fitted slope b>0, we set b=0 (no correction),
    because photobleaching should be a decay, not an increase.
    """
    y = np.asarray(y, float)
    t = np.asarray(timepoints, float)

    m = np.isfinite(y) & (y > 0)
    if m.sum() < 3:
        # can't fit -> constant (no correction)
        return np.ones_like(t, float)

    logy = np.log(np.clip(y[m], EPS, None))
    b, a = np.polyfit(t[m], logy, 1)  # logy ≈ a + b t

    if enforce_decay_only and (b > 0):
        b = 0.0

    fitted = np.exp(a + b * t)
    return fitted


def make_corr_factor(fitted: np.ndarray, *, max_corr: float = MAX_CORR_FACTOR) -> np.ndarray:
    """Return multiplicative correction factor: corr[t] = fitted[t0] / fitted[t].
    Clipped to max_corr, NaN-safe.
    """
    fitted = np.asarray(fitted, float)
    ok = np.isfinite(fitted) & (fitted > 0)
    if not np.any(ok):
        return np.ones_like(fitted, float)

    # normalize at first finite point
    f0 = float(fitted[ok][0])
    corr = f0 / np.clip(fitted, EPS, None)

    # clip / sanitize
    corr = np.nan_to_num(corr, nan=1.0, posinf=max_corr, neginf=1.0)
    corr = np.minimum(corr, max_corr)
    corr = np.maximum(corr, 0.0)
    return corr


def needs_bleach_correction_auto(ref: np.ndarray, *, thr: float = MCP_DECAY_THRESHOLD) -> bool:
    """Heuristic: apply correction if median(head)/median(tail) >= 1+thr (i.e., clear decay)."""
    ref = np.asarray(ref, float)
    if np.isfinite(ref).sum() < 6:
        return False
    k = max(3, len(ref) // 10)  # first/last 10% (>=3)
    head = np.nanmedian(ref[:k])
    tail = np.nanmedian(ref[-k:])
    if (not np.isfinite(head)) or (not np.isfinite(tail)) or head <= 0 or tail <= 0:
        return False
    ratio = head / tail
    return ratio >= (1.0 + thr)


def extract_patch(img2d: np.ndarray, cy: float, cx: float, r: int):
    """Extract (2r+1)x(2r+1) patch around (cy,cx). Returns None if out-of-bounds."""
    H, W = img2d.shape
    cyi, cxi = int(round(cy)), int(round(cx))
    y0, y1 = cyi - r, cyi + r + 1
    x0, x1 = cxi - r, cxi + r + 1
    if y0 < 0 or x0 < 0 or y1 > H or x1 > W:
        return None
    return img2d[y0:y1, x0:x1]


def radial_profile(img2d: np.ndarray, *, center=None, r_max=None):
    """Mean intensity in integer-radius rings."""
    H, W = img2d.shape
    if center is None:
        cy, cx = (H - 1) / 2.0, (W - 1) / 2.0
    else:
        cy, cx = center

    yy, xx = np.indices((H, W))
    rr = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    if r_max is None:
        r_max = int(np.floor(rr.max()))
    r_int = np.clip(rr.astype(int), 0, r_max)

    sums = np.bincount(r_int.ravel(), weights=img2d.ravel(), minlength=r_max + 1)
    cnts = np.bincount(r_int.ravel(), minlength=r_max + 1)
    prof = sums / np.maximum(cnts, 1)
    return np.arange(r_max + 1), prof


# --------------------------
# Main
# --------------------------
def run_cell6():
    cell_folders = sorted([p for p in TRANS_DIR.iterdir() if p.is_dir()])
    print(f"Found {len(cell_folders)} cells with transcription outputs in: {TRANS_DIR}")

    all_rows = []

    # Cumulative overlay for on/off (2D MIP patch centered at TetR)
    acc = {
        "on":  {CH_SNAP: None, CH_MCP: None, CH_TETR: None, "n": 0},
        "off": {CH_SNAP: None, CH_MCP: None, CH_TETR: None, "n": 0},
    }
    patch_shape = (2 * PATCH_R + 1, 2 * PATCH_R + 1)

    for cell_folder in tqdm(cell_folders, desc="Cell6 per-cell"):
        cell_id = cell_folder.name
        raw_path = RAW_DIR / f"{cell_id}.tif"
        trans_csv = cell_folder / "transcription_status.csv"
        refined_cell_dir = REFINED_DIR / cell_id
        mask_path = refined_cell_dir / "mask.npy"

        if not (raw_path.exists() and trans_csv.exists() and mask_path.exists()):
            continue

        img_5d = imread(raw_path)     # (T,Z,C,Y,X)
        masks = np.load(mask_path)    # (T,Y,X)
        df = pd.read_csv(trans_csv)

        T = int(img_5d.shape[0])
        T = min(T, T_EXPECT)

        # 1) reference mean intensities (SNAP + MCP) from nucleus mask (fallback: cell mask)
        snap_ref = np.full(T, np.nan, float)
        mcp_ref  = np.full(T, np.nan, float)

        for t in range(T):
            snap_2d = img_5d[t, :, CH_SNAP].max(axis=0)
            mcp_2d  = img_5d[t, :, CH_MCP].max(axis=0)

            cell_mask = masks[t].astype(bool)
            nuc_mask = get_nucleus_mask(snap_2d, cell_mask)
            ref_mask = nuc_mask if np.any(nuc_mask) else cell_mask

            if np.any(ref_mask):
                snap_ref[t] = float(np.mean(snap_2d[ref_mask]))
                mcp_ref[t]  = float(np.mean(mcp_2d[ref_mask]))

        tvec = np.arange(T, dtype=float)

        # 2) SNAP correction
        snap_fitted = fit_exponential_decay(tvec, snap_ref, enforce_decay_only=ENFORCE_DECAY_ONLY)
        snap_corr = make_corr_factor(snap_fitted, max_corr=MAX_CORR_FACTOR)
        snap_ref_corr = snap_ref * snap_corr

        # 3) MCP correction (optional/auto)
        mcp_fitted = fit_exponential_decay(tvec, mcp_ref, enforce_decay_only=ENFORCE_DECAY_ONLY)
        mcp_corr_raw = make_corr_factor(mcp_fitted, max_corr=MAX_CORR_FACTOR)

        if APPLY_MCP_BLEACH_CORR is True:
            apply_mcp = True
        elif APPLY_MCP_BLEACH_CORR is False:
            apply_mcp = False
        else:  # "auto"
            apply_mcp = needs_bleach_correction_auto(mcp_ref, thr=MCP_DECAY_THRESHOLD)

        mcp_corr = mcp_corr_raw if apply_mcp else np.ones_like(mcp_corr_raw)
        mcp_ref_corr = mcp_ref * mcp_corr

        # 4) Plot bleaching reference curves (SNAP always; MCP shown too)
        fig = plt.figure(figsize=(7, 4))
        plt.plot(tvec, snap_ref, color="blue", alpha=0.7, label="SNAP ref raw")
        plt.plot(tvec, snap_ref_corr, color="deepskyblue", label="SNAP ref corrected")
        plt.plot(tvec, mcp_ref, color="red", alpha=0.35, label="MCP ref raw")
        plt.plot(tvec, mcp_ref_corr, color="darkred", alpha=0.7,
                 label=f"MCP ref corrected ({'ON' if apply_mcp else 'OFF'})")
        plt.xlabel("t (frame)")
        plt.ylabel("reference mean intensity (a.u.)")
        plt.title(f"{cell_id} bleaching reference (corr cap={MAX_CORR_FACTOR:g})")
        plt.legend(fontsize=8)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "per_cell_plots" / f"{cell_id}_bleach_ref.png", dpi=150)
        plt.close(fig)

        # 5) Build per-cell dataframe for this cell (t in [0..T-1])
        df_local = df[df["t"].between(0, T - 1)].copy().sort_values("t").reset_index(drop=True)
        df_local["cell_id"] = cell_id

        # ensure numeric
        df_local["snap_roi_mean"] = pd.to_numeric(df_local.get("snap_roi_mean", np.nan), errors="coerce")
        df_local["mcp_roi_mean"]  = pd.to_numeric(df_local.get("mcp_roi_mean", np.nan), errors="coerce")
        if "tetr_intensity" in df_local.columns:
            df_local["tetr_intensity"] = pd.to_numeric(df_local["tetr_intensity"], errors="coerce")

        # per-row correction factors
        idx = df_local["t"].to_numpy(dtype=int)
        df_local["ch1_corr_factor"] = snap_corr[idx]
        df_local["ch2_corr_factor"] = mcp_corr[idx]
        df_local["ch2_bleach_applied"] = int(apply_mcp)

        # standard columns expected by downstream analysis
        df_local["ch1_mean"] = df_local["snap_roi_mean"]
        df_local["ch2_mean"] = df_local["mcp_roi_mean"]
        df_local["ch3_mean"] = df_local["tetr_intensity"] if "tetr_intensity" in df_local.columns else np.nan

        # *_detrended == bleaching-corrected (compat naming)
        df_local["ch1_detrended"] = df_local["ch1_mean"] * df_local["ch1_corr_factor"]
        df_local["ch2_detrended"] = df_local["ch2_mean"] * df_local["ch2_corr_factor"]

        # also keep legacy names (useful for QA/debug)
        df_local["snap_ref_mean"] = snap_ref[idx]
        df_local["snap_ref_mean_corrected"] = snap_ref_corr[idx]
        df_local["mcp_ref_mean"] = mcp_ref[idx]
        df_local["mcp_ref_mean_corrected"] = mcp_ref_corr[idx]

        # 6) Per-cell plots used in the original Cell6 notebook
        # Plot 1: MCP vs SNAP (both corrected)
        fig = plt.figure(figsize=(7, 4))
        ax1 = plt.gca()
        ax2 = ax1.twinx()
        ax1.plot(df_local["t"], df_local["ch2_detrended"], color="red", label="MCP ROI mean (corr)")
        ax2.plot(df_local["t"], df_local["ch1_detrended"], color="blue", label="SNAP ROI mean (corr)")
        ax1.set_xlabel("t (frame)")
        ax1.set_ylabel("MCP ROI mean (a.u.)")
        ax2.set_ylabel("SNAP ROI mean (a.u.)")
        ax1.set_title(f"{cell_id}: MCP & SNAP (bleach-corrected)")
        ax1.legend(loc="upper left", fontsize=8)
        ax2.legend(loc="upper right", fontsize=8)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "per_cell_plots" / f"{cell_id}_mcp_vs_snapROI.png", dpi=150)
        plt.close(fig)

        # Plot 2: TetR (raw) vs SNAP (corrected)
        fig = plt.figure(figsize=(7, 4))
        ax1 = plt.gca()
        ax2 = ax1.twinx()
        if "tetr_intensity" in df_local.columns:
            ax1.plot(df_local["t"], df_local["tetr_intensity"], color="green", label="TetR intensity (raw)")
        ax2.plot(df_local["t"], df_local["ch1_detrended"], color="blue", label="SNAP ROI mean (corr)")
        ax1.set_xlabel("t (frame)")
        ax1.set_ylabel("TetR intensity (a.u.)")
        ax2.set_ylabel("SNAP ROI mean (a.u.)")
        ax1.set_title(f"{cell_id}: TetR (raw) & SNAP (bleach-corrected)")
        ax1.legend(loc="upper left", fontsize=8)
        ax2.legend(loc="upper right", fontsize=8)
        plt.tight_layout()
        plt.savefig(OUT_DIR / "per_cell_plots" / f"{cell_id}_tetr_vs_snapROI.png", dpi=150)
        plt.close(fig)

        all_rows.append(df_local)

        # 7) ON/OFF patch accumulation (center at TetR)
        for _, r in df_local.iterrows():
            state = str(r.get("state", "")).lower()
            if state not in ("on", "off"):
                continue
            if np.isnan(r.get("tetr_y", np.nan)) or np.isnan(r.get("tetr_x", np.nan)):
                continue

            t = int(r["t"])
            cy, cx = float(r["tetr_y"]), float(r["tetr_x"])

            snap2d = img_5d[t, :, CH_SNAP].max(axis=0)
            mcp2d  = img_5d[t, :, CH_MCP].max(axis=0)
            tetr2d = img_5d[t, :, CH_TETR].max(axis=0)

            p1 = extract_patch(snap2d, cy, cx, PATCH_R)
            p2 = extract_patch(mcp2d,  cy, cx, PATCH_R)
            p3 = extract_patch(tetr2d, cy, cx, PATCH_R)
            if p1 is None or p2 is None or p3 is None:
                continue

            # apply per-frame correction to channels 1/2 before accumulating
            p1c = p1.astype(float) * float(snap_corr[t])
            p2c = p2.astype(float) * float(mcp_corr[t])
            p3c = p3.astype(float)  # TetR unchanged (extend if needed)

            for ch in (CH_SNAP, CH_MCP, CH_TETR):
                if acc[state][ch] is None:
                    acc[state][ch] = np.zeros(patch_shape, dtype=float)

            acc[state][CH_SNAP] += p1c
            acc[state][CH_MCP]  += p2c
            acc[state][CH_TETR] += p3c
            acc[state]["n"] += 1

    # --------------------------
    # Save aggregated outputs
    # --------------------------
    if len(all_rows) == 0:
        print("No cells processed (no transcription outputs found or missing inputs).")
        return

    all_df = pd.concat(all_rows, ignore_index=True)

    # ensure "state" column exists
    if "state" in all_df.columns:
        all_df["state"] = all_df["state"].astype(str).str.strip()
    else:
        all_df["state"] = "OFF"

    on_df  = all_df[all_df["state"].str.lower() == "on"].copy()
    off_df = all_df[all_df["state"].str.lower() == "off"].copy()

    on_df.to_csv(OUT_DIR / "all_on.csv", index=False)
    off_df.to_csv(OUT_DIR / "all_off.csv", index=False)

    # merged CSV for downstream analysis
    out = pd.DataFrame()
    out["frame"] = pd.to_numeric(all_df.get("t", np.nan), errors="coerce").astype("Int64")
    out["cell_id"] = all_df["cell_id"].astype(str)
    out["state"] = all_df["state"].astype(str).str.upper()

    # ref_confidence (if exists)
    if "ref_confidence" in all_df.columns:
        out["ref_confidence"] = pd.to_numeric(all_df["ref_confidence"], errors="coerce").fillna(1).astype(float)
    elif "confidence" in all_df.columns:
        out["ref_confidence"] = pd.to_numeric(all_df["confidence"], errors="coerce").fillna(1).astype(float)
    else:
        out["ref_confidence"] = 1.0

    # Required base signals (RAW)
    out["ch1_mean"] = pd.to_numeric(all_df.get("ch1_mean", np.nan), errors="coerce")
    out["ch2_mean"] = pd.to_numeric(all_df.get("ch2_mean", np.nan), errors="coerce")
    out["ch3_mean"] = pd.to_numeric(all_df.get("ch3_mean", np.nan), errors="coerce")

    # Bleach-corrected ("detrended" naming kept for compatibility)
    out["ch1_detrended"] = pd.to_numeric(all_df.get("ch1_detrended", np.nan), errors="coerce")
    out["ch2_detrended"] = pd.to_numeric(all_df.get("ch2_detrended", np.nan), errors="coerce")

    # Optional provenance columns (useful for QC / reproducibility)
    out["ch1_corr_factor"] = pd.to_numeric(all_df.get("ch1_corr_factor", np.nan), errors="coerce")
    out["ch2_corr_factor"] = pd.to_numeric(all_df.get("ch2_corr_factor", np.nan), errors="coerce")
    out["ch2_bleach_applied"] = pd.to_numeric(all_df.get("ch2_bleach_applied", np.nan), errors="coerce")

    # sort by cell_id, then frame
    out = out.sort_values(["cell_id", "frame"]).reset_index(drop=True)

    out_path = OUT_DIR / "all_cells_spot_intensity_seg_cell.csv"
    out.to_csv(out_path, index=False)
    print(f"Wrote merged CSV: {out_path}")

    # --------------------------
    # Global heatmaps + radial profiles (OFF vs ON in one figure)
    # --------------------------
    channel_specs = [
        (CH_SNAP, "ch1_SNAP"),
        (CH_MCP,  "ch2_MCP"),
        (CH_TETR, "ch3_TetR"),
    ]

    for ch, ch_name in channel_specs:
        n_on = int(acc["on"]["n"])
        n_off = int(acc["off"]["n"])

        mean_on = (acc["on"][ch] / n_on) if (n_on > 0 and acc["on"][ch] is not None) else None
        mean_off = (acc["off"][ch] / n_off) if (n_off > 0 and acc["off"][ch] is not None) else None

        if mean_on is None and mean_off is None:
            print(f"No patches for {ch_name}.")
            continue

        imgs_for_lim = []
        if mean_on is not None:
            imgs_for_lim.append(mean_on)
        if mean_off is not None:
            imgs_for_lim.append(mean_off)
        all_vals = np.concatenate([im.ravel() for im in imgs_for_lim])
        vmin, vmax = float(np.min(all_vals)), float(np.max(all_vals))

        # Heatmap: OFF vs ON (shared colorbar)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

        if mean_off is not None:
            im0 = axes[0].imshow(mean_off, cmap="gray", vmin=vmin, vmax=vmax)
            axes[0].set_title(f"OFF {ch_name} (n={n_off})")
        else:
            im0 = axes[0].imshow(np.zeros(patch_shape), cmap="gray", vmin=vmin, vmax=vmax)
            axes[0].set_title(f"OFF {ch_name} (n=0)")
        axes[0].axis("off")

        if mean_on is not None:
            im1 = axes[1].imshow(mean_on, cmap="gray", vmin=vmin, vmax=vmax)
            axes[1].set_title(f"ON {ch_name} (n={n_on})")
        else:
            im1 = axes[1].imshow(np.zeros(patch_shape), cmap="gray", vmin=vmin, vmax=vmax)
            axes[1].set_title(f"ON {ch_name} (n=0)")
        axes[1].axis("off")

        # One shared colorbar (use im1 unless mean_on is None)
        mappable = im1 if mean_on is not None else im0
        cbar = fig.colorbar(mappable, ax=axes, shrink=0.9)
        cbar.set_label("intensity (a.u.)")

        fig.suptitle(f"{ch_name}: OFF vs ON (shared colorbar)")
        plt.savefig(OUT_DIR / "global" / f"{ch_name}_OFF_ON_heatmap.png", dpi=200)
        plt.close(fig)

        # Radial profile: ON (red) vs OFF (blue)
        fig = plt.figure(figsize=(6, 4))
        if mean_off is not None:
            rr, prof_off = radial_profile(mean_off, center=(PATCH_R, PATCH_R), r_max=PATCH_R)
            plt.plot(rr, prof_off, color="blue", label=f"OFF (n={n_off})")
        if mean_on is not None:
            rr, prof_on = radial_profile(mean_on, center=(PATCH_R, PATCH_R), r_max=PATCH_R)
            plt.plot(rr, prof_on, color="red", label=f"ON (n={n_on})")

        plt.xlabel("radius (px)")
        plt.ylabel("mean intensity (a.u.)")
        plt.title(f"{ch_name} radial profile (ON red, OFF blue)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(OUT_DIR / "global" / f"{ch_name}_radial_ON_vs_OFF.png", dpi=200)
        plt.close(fig)

    print(f"Cell6 done. Outputs saved to: {OUT_DIR}")


if __name__ == "__main__":
    run_cell6()
